# India TB Project — MySQL SQL Queries
**Run AFTER the main notebook which uploads data to MySQL**

Two tables are available:
- `tb_who` — Year, TB_Incidence, Treatment_Success (time series)
- `tb_tobacco` — State, all tobacco-TB columns (single year snapshot)

This notebook runs 10 SQL queries covering both tables.


In [1]:
import mysql.connector
import pandas as pd

conn = mysql.connector.connect(
    host='127.0.0.1',
    user='root',
    password='piku2005',   # UPDATE THIS
    database='tb_india'
)
print('Connected to MySQL!')

def run_query(sql):
    return pd.read_sql_query(sql, conn)

# Check both tables
print('\ntb_who preview:')
display(run_query('SELECT * FROM tb_who LIMIT 5'))
print('\ntb_tobacco preview:')
display(run_query('SELECT State, Tobacco_TB_Total_Pct, Cessation_Total_Pct FROM tb_tobacco LIMIT 5'))


Connected to MySQL!

tb_who preview:


C:\Users\KIIT0001\AppData\Local\Temp\ipykernel_14104\2731819608.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql, conn)


,id,Year,TB_Incidence,Treatment_Success
0,1,2000,180.5,52.0
1,2,2001,177.0,56.0
2,3,2002,174.0,66.0
3,4,2003,171.0,73.0
4,5,2004,168.5,77.0



tb_tobacco preview:


C:\Users\KIIT0001\AppData\Local\Temp\ipykernel_14104\2731819608.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql, conn)


,State,Tobacco_TB_Total_Pct,Cessation_Total_Pct
0,Andaman & Nicobar Islands,90.9,33.7
1,Andhra Pradesh,96.8,24.9
2,Arunachal Pradesh,74.9,56.4
3,Assam,71.4,19.3
4,Bihar,48.5,18.1



## Query 1: TB Incidence Ranked by Year
**Which year had the highest TB burden?**

In [2]:
q1 = '''
SELECT
    Year,
    ROUND(TB_Incidence, 2)                                    AS TB_Incidence,
    RANK() OVER (ORDER BY TB_Incidence DESC)                  AS burden_rank,
    ROUND(TB_Incidence - FIRST_VALUE(TB_Incidence)
          OVER (ORDER BY Year), 2)                            AS change_from_start
FROM tb_who
WHERE TB_Incidence IS NOT NULL
ORDER BY Year
'''
run_query(q1)


C:\Users\KIIT0001\AppData\Local\Temp\ipykernel_14104\2731819608.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql, conn)


,Year,TB_Incidence,burden_rank,change_from_start
0,2000,180.50,2,0.00
1,2001,177.00,3,-3.50
2,2002,174.00,4,-6.50
3,2003,171.00,5,-9.50
4,2004,168.50,6,-12.00
5,2005,166.00,7,-14.50
6,2006,163.00,8,-17.50
7,2007,158.50,9,-22.00
8,2008,155.00,10,-25.50
9,2009,150.00,11,-30.50



## Query 2: Year-over-Year Change Using LAG()
**How much did TB drop each year?**

In [3]:
q2 = '''
WITH yearly AS (
    SELECT Year, ROUND(AVG(TB_Incidence), 2) AS avg_incidence
    FROM tb_who WHERE TB_Incidence IS NOT NULL
    GROUP BY Year
)
SELECT
    Year,
    avg_incidence,
    LAG(avg_incidence) OVER (ORDER BY Year)                             AS prev_year,
    ROUND(avg_incidence - LAG(avg_incidence) OVER (ORDER BY Year), 2)  AS yoy_change,
    ROUND((avg_incidence - LAG(avg_incidence) OVER (ORDER BY Year))
          / LAG(avg_incidence) OVER (ORDER BY Year) * 100, 2)          AS yoy_pct_change
FROM yearly
ORDER BY Year
'''
run_query(q2)


C:\Users\KIIT0001\AppData\Local\Temp\ipykernel_14104\2731819608.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql, conn)


,Year,avg_incidence,prev_year,yoy_change,yoy_pct_change
0,2000,180.50,NaN,NaN,NaN
1,2001,177.00,180.50,-3.50,-1.94
2,2002,174.00,177.00,-3.00,-1.69
3,2003,171.00,174.00,-3.00,-1.72
4,2004,168.50,171.00,-2.50,-1.46
5,2005,166.00,168.50,-2.50,-1.48
6,2006,163.00,166.00,-3.00,-1.81
7,2007,158.50,163.00,-4.50,-2.76
8,2008,155.00,158.50,-3.50,-2.21
9,2009,150.00,155.00,-5.00,-3.23


## Query 3: 5-Year Rolling Average Trend
**Smooth out noise to see the real trend**

In [4]:
q3 = '''
SELECT
    Year,
    ROUND(TB_Incidence, 2) AS TB_Incidence,
    ROUND(AVG(TB_Incidence) OVER (
        ORDER BY Year ROWS BETWEEN 2 PRECEDING AND 2 FOLLOWING
    ), 2)                  AS rolling_5yr_avg
FROM tb_who
WHERE TB_Incidence IS NOT NULL
ORDER BY Year
'''
run_query(q3)


C:\Users\KIIT0001\AppData\Local\Temp\ipykernel_14104\2731819608.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql, conn)


,Year,TB_Incidence,rolling_5yr_avg
0,2000,180.50,177.17
1,2001,177.00,175.62
2,2002,174.00,174.20
3,2003,171.00,171.30
4,2004,168.50,168.50
5,2005,166.00,165.40
6,2006,163.00,162.20
7,2007,158.50,158.50
8,2008,155.00,154.50
9,2009,150.00,150.20



## Query 4: Classify Each Year by TB Level Using CASE WHEN

In [5]:
q4 = '''
SELECT
    Year,
    ROUND(TB_Incidence, 2) AS TB_Incidence,
    CASE
        WHEN TB_Incidence >= 300 THEN 'CRITICAL'
        WHEN TB_Incidence >= 200 THEN 'HIGH'
        WHEN TB_Incidence >= 100 THEN 'MODERATE'
        WHEN TB_Incidence <  100 THEN 'LOW'
        ELSE 'UNKNOWN'
    END AS burden_level
FROM tb_who
WHERE TB_Incidence IS NOT NULL
ORDER BY Year
'''
run_query(q4)


C:\Users\KIIT0001\AppData\Local\Temp\ipykernel_14104\2731819608.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql, conn)


,Year,TB_Incidence,burden_level
0,2000,180.50,MODERATE
1,2001,177.00,MODERATE
2,2002,174.00,MODERATE
3,2003,171.00,MODERATE
4,2004,168.50,MODERATE
5,2005,166.00,MODERATE
6,2006,163.00,MODERATE
7,2007,158.50,MODERATE
8,2008,155.00,MODERATE
9,2009,150.00,MODERATE



## Query 5: Treatment Success Rate — Years Meeting WHO 90% Target

In [6]:
q5 = '''
SELECT
    Year,
    ROUND(Treatment_Success, 2)  AS Treatment_Success,
    CASE
        WHEN Treatment_Success >= 90 THEN 'Meets WHO Target'
        WHEN Treatment_Success >= 80 THEN 'Near Target'
        ELSE 'Below Target'
    END AS who_status,
    ROUND(90 - Treatment_Success, 2) AS gap_to_90pct
FROM tb_who
WHERE Treatment_Success IS NOT NULL
ORDER BY Year
'''
run_query(q5)


C:\Users\KIIT0001\AppData\Local\Temp\ipykernel_14104\2731819608.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql, conn)


,Year,Treatment_Success,who_status,gap_to_90pct
0,2000,52.00,Below Target,38.00
1,2001,56.00,Below Target,34.00
2,2002,66.00,Below Target,24.00
3,2003,73.00,Below Target,17.00
4,2004,77.00,Below Target,13.00
5,2005,79.00,Below Target,11.00
6,2006,79.50,Below Target,10.50
7,2007,68.33,Below Target,21.67
8,2008,69.50,Below Target,20.50
9,2009,72.00,Below Target,18.00



## Query 6: States Ranked by Tobacco-TB Burden
*(from tb_tobacco table)*

In [7]:
q6 = '''
SELECT
    State,
    ROUND(Tobacco_TB_Total_Pct, 2)                                     AS tobacco_tb_pct,
    RANK() OVER (ORDER BY Tobacco_TB_Total_Pct DESC)                   AS tobacco_rank,
    CASE
        WHEN Tobacco_TB_Total_Pct >= 75 THEN 'CRITICAL'
        WHEN Tobacco_TB_Total_Pct >= 50 THEN 'HIGH'
        WHEN Tobacco_TB_Total_Pct >= 25 THEN 'MEDIUM'
        ELSE 'LOW'
    END AS tobacco_burden_level
FROM tb_tobacco
WHERE Tobacco_TB_Total_Pct IS NOT NULL
ORDER BY tobacco_tb_pct DESC
'''
run_query(q6)


C:\Users\KIIT0001\AppData\Local\Temp\ipykernel_14104\2731819608.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql, conn)


,State,tobacco_tb_pct,tobacco_rank,tobacco_burden_level
0,Lakshadweep,100.0,1,CRITICAL
1,Mizoram,97.9,2,CRITICAL
2,Himachal Pradesh,97.4,3,CRITICAL
3,Puducherry,97.1,4,CRITICAL
4,Andhra Pradesh,96.8,5,CRITICAL
5,Gujarat,94.5,6,CRITICAL
6,Uttarakhand,94.4,7,CRITICAL
7,Odisha,94.4,7,CRITICAL
8,Chandigarh,94.0,9,CRITICAL
9,Telangana,92.8,10,CRITICAL



## Query 7: Cessation Coverage Gap by State
**Which states have highest % of tobacco users getting NO quit-smoking support?**

In [8]:
q7 = '''
SELECT
    State,
    ROUND(Tobacco_Users_Total_Count, 0)                            AS tobacco_users,
    ROUND(Cessation_Total_Count, 0)                                AS linked_to_cessation,
    ROUND(Tobacco_Users_Total_Count - Cessation_Total_Count, 0)   AS NOT_linked,
    ROUND(
        (Tobacco_Users_Total_Count - Cessation_Total_Count)
        / NULLIF(Tobacco_Users_Total_Count, 0) * 100
    , 1)                                                           AS cessation_gap_pct,
    RANK() OVER (
        ORDER BY (Tobacco_Users_Total_Count - Cessation_Total_Count)
        / NULLIF(Tobacco_Users_Total_Count, 0) DESC
    )                                                              AS gap_rank
FROM tb_tobacco
WHERE Tobacco_Users_Total_Count IS NOT NULL
  AND Cessation_Total_Count IS NOT NULL
ORDER BY cessation_gap_pct DESC
'''
run_query(q7)


C:\Users\KIIT0001\AppData\Local\Temp\ipykernel_14104\2731819608.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql, conn)


,State,tobacco_users,linked_to_cessation,NOT_linked,cessation_gap_pct,gap_rank
0,Sikkim,79.0,6.0,73.0,92.4,1
1,Goa,106.0,9.0,97.0,91.5,2
2,Tripura,478.0,44.0,434.0,90.8,3
3,Punjab,1778.0,282.0,1496.0,84.1,4
4,Meghalaya,1444.0,257.0,1187.0,82.2,5
5,Bihar,5291.0,958.0,4333.0,81.9,6
6,Madhya Pradesh,14599.0,2789.0,11810.0,80.9,7
7,Assam,8254.0,1590.0,6664.0,80.7,8
8,Chandigarh,307.0,60.0,247.0,80.5,9
9,Delhi,3456.0,782.0,2674.0,77.4,10



## Query 8: Public vs Private Sector Split by State

In [9]:
q8 = '''
SELECT
    State,
    ROUND(Tobacco_TB_Public_Count, 0)   AS public_count,
    ROUND(Tobacco_TB_Private_Count, 0)  AS private_count,
    ROUND(Tobacco_TB_Total_Count, 0)    AS total_count,
    ROUND(
        Tobacco_TB_Public_Count
        / NULLIF(Tobacco_TB_Total_Count, 0) * 100
    , 1)                                AS public_share_pct,
    CASE
        WHEN Tobacco_TB_Public_Count > Tobacco_TB_Private_Count THEN 'Mostly Public'
        WHEN Tobacco_TB_Private_Count > Tobacco_TB_Public_Count THEN 'Mostly Private'
        ELSE 'Equal'
    END AS dominant_sector
FROM tb_tobacco
WHERE Tobacco_TB_Total_Count IS NOT NULL
ORDER BY total_count DESC
'''
run_query(q8)


C:\Users\KIIT0001\AppData\Local\Temp\ipykernel_14104\2731819608.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql, conn)


,State,public_count,private_count,total_count,public_share_pct,dominant_sector
0,India,1473433.0,354179.0,1827612.0,80.6,Mostly Public
1,Uttar Pradesh,301001.0,57695.0,358696.0,83.9,Mostly Public
2,Maharashtra,128562.0,53968.0,182530.0,70.4,Mostly Public
3,Gujarat,104192.0,35780.0,139972.0,74.4,Mostly Public
4,Rajasthan,106155.0,28475.0,134630.0,78.8,Mostly Public
5,Madhya Pradesh,81505.0,20132.0,101637.0,80.2,Mostly Public
6,Andhra Pradesh,63627.0,26477.0,90104.0,70.6,Mostly Public
7,West Bengal,80975.0,6168.0,87143.0,92.9,Mostly Public
8,Tamil Nadu,71409.0,12182.0,83591.0,85.4,Mostly Public
9,Bihar,44248.0,36165.0,80413.0,55.0,Mostly Public



## Query 9: Cross-Table — States With High Tobacco Burden (used as priority filter)

In [10]:
q9 = '''
SELECT
    t.State,
    ROUND(t.Tobacco_TB_Total_Pct, 2)    AS tobacco_tb_pct,
    ROUND(t.Cessation_Total_Pct, 2)     AS cessation_support_pct,
    ROUND(
        100 - t.Cessation_Total_Pct
    , 2)                                 AS cessation_gap_pct,
    CASE
        WHEN t.Tobacco_TB_Total_Pct >= 50
         AND (100 - t.Cessation_Total_Pct) >= 50
        THEN 'HIGH RISK — High Tobacco + Poor Cessation'
        WHEN t.Tobacco_TB_Total_Pct >= 50
        THEN 'MODERATE — High Tobacco but Cessation Covered'
        ELSE 'LOW RISK'
    END AS priority_label
FROM tb_tobacco t
WHERE t.Tobacco_TB_Total_Pct IS NOT NULL
ORDER BY tobacco_tb_pct DESC
'''
run_query(q9)


C:\Users\KIIT0001\AppData\Local\Temp\ipykernel_14104\2731819608.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql, conn)


,State,tobacco_tb_pct,cessation_support_pct,cessation_gap_pct,priority_label
0,Lakshadweep,100.0,100.0,0.0,MODERATE — High Tobacco but Cessation Covered
1,Mizoram,97.9,89.0,11.0,MODERATE — High Tobacco but Cessation Covered
2,Himachal Pradesh,97.4,90.1,9.9,MODERATE — High Tobacco but Cessation Covered
3,Puducherry,97.1,98.2,1.8,MODERATE — High Tobacco but Cessation Covered
4,Andhra Pradesh,96.8,24.9,75.1,HIGH RISK — High Tobacco + Poor Cessation
5,Gujarat,94.5,41.7,58.3,HIGH RISK — High Tobacco + Poor Cessation
6,Uttarakhand,94.4,31.3,68.7,HIGH RISK — High Tobacco + Poor Cessation
7,Odisha,94.4,44.7,55.3,HIGH RISK — High Tobacco + Poor Cessation
8,Chandigarh,94.0,19.5,80.5,HIGH RISK — High Tobacco + Poor Cessation
9,Telangana,92.8,53.3,46.7,MODERATE — High Tobacco but Cessation Covered



## Query 10: Summary Statistics for Both Tables Using CTE

In [11]:
q10 = '''
WITH who_stats AS (
    SELECT
        'WHO TB Time Series'          AS dataset,
        COUNT(*)                       AS total_rows,
        MIN(Year)                      AS start_year,
        MAX(Year)                      AS end_year,
        ROUND(AVG(TB_Incidence), 2)    AS avg_incidence,
        ROUND(MIN(TB_Incidence), 2)    AS min_incidence,
        ROUND(MAX(TB_Incidence), 2)    AS max_incidence
    FROM tb_who
    WHERE TB_Incidence IS NOT NULL
),
tobacco_stats AS (
    SELECT
        'TB Tobacco State Data'        AS dataset,
        COUNT(*)                       AS total_rows,
        NULL                           AS start_year,
        NULL                           AS end_year,
        ROUND(AVG(Tobacco_TB_Total_Pct), 2) AS avg_incidence,
        ROUND(MIN(Tobacco_TB_Total_Pct), 2) AS min_incidence,
        ROUND(MAX(Tobacco_TB_Total_Pct), 2) AS max_incidence
    FROM tb_tobacco
    WHERE Tobacco_TB_Total_Pct IS NOT NULL
)
SELECT * FROM who_stats
UNION ALL
SELECT * FROM tobacco_stats
'''
run_query(q10)


C:\Users\KIIT0001\AppData\Local\Temp\ipykernel_14104\2731819608.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql, conn)


,dataset,total_rows,start_year,end_year,avg_incidence,min_incidence,max_incidence
0,WHO TB Time Series,25,2000.0,2024.0,139.53,97.45,187.0
1,TB Tobacco State Data,37,NaN,NaN,81.55,48.50,100.0



## SQL Summary

| Query | Table | Technique |
|---|---|---|
| Q1 — Incidence ranked | tb_who | RANK(), FIRST_VALUE() OVER |
| Q2 — YoY change | tb_who | CTE, LAG(), % change |
| Q3 — Rolling 5yr avg | tb_who | AVG() OVER ROWS BETWEEN |
| Q4 — Year classification | tb_who | CASE WHEN |
| Q5 — WHO target status | tb_who | CASE WHEN, gap calculation |
| Q6 — State tobacco rank | tb_tobacco | RANK() OVER, CASE WHEN |
| Q7 — Cessation gap | tb_tobacco | NULLIF, RANK() OVER |
| Q8 — Public vs Private | tb_tobacco | NULLIF, CASE WHEN |
| Q9 — Priority states | tb_tobacco | Nested CASE WHEN |
| Q10 — Summary stats | Both | CTE, UNION ALL |


In [12]:
conn.close()
print('MySQL connection closed.')

MySQL connection closed.
